# Capstone — mirrors the deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamzaviour/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Paper question:** *Which content pages should a team review first, and how much of a decline-risk ranking survives an honest test?*

This notebook collects the receipts (metrics JSONs from Weeks 4–7) and regenerates every chart the paper embeds. Nothing here is re-fitted: each number on the page traces to a committed file in `work/outputs/`. Wording is observed / measured / directional / decision-support.

In [1]:
import os, json, numpy as np, pandas as pd, shutil
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
if os.path.exists("../../data/raw/content_refresh_anonymized.csv"): os.chdir("../..")
os.makedirs("work/figures", exist_ok=True); os.makedirs("docs/img", exist_ok=True)
J = lambda n: json.load(open(f"work/outputs/{n}.json"))
w5, w6, w7, w4 = J("w05_model_metrics"), J("w06_validation_audit_metrics"), J("action_playbook_metrics"), J("baseline_action_score_metrics")
BASE = w7["base_rate_declining"]
INK, BLUE, GREY, RED, GREEN = "#1f2933", "#2f6f9f", "#9aa5b1", "#b8433a", "#3a8f6b"
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": GREY})
def save(fig, name):
    fig.tight_layout(); fig.savefig(f"work/figures/{name}.png", dpi=160); shutil.copy(f"work/figures/{name}.png", f"docs/img/{name}.png"); plt.close(fig)
print("base rate (declining):", BASE)

base rate (declining): 0.5421


## 1. Question

A content team can review only a few dozen pages per sprint. The decision: *which pages first, and why?* Target used for modelling: a page is "declining" when its impressions in the last 30 days fell by more than 20% against the 30 days before (the release's `trend_direction == "down"`). Base rate in the snapshot: 54.2%.

## 2. Data

- **Modelling snapshot (starter slice):** 30,000 pages, 44 columns, 32 pseudonymised clients, trailing 90 days. All modelling, the baseline, the audit and the playbook use this file.
- **Warehouse check (release v20260703, hosted on Hugging Face):** I verified the data contract on one mid-panel partition, `month=2026-03`, with three queries (grain, row count and span, availability with `IS TRUE`) and ran a leakage demonstration on it (`w03_data_contract.ipynb`). June 2026 (the sealed final month) was never touched. I did **not** scan the whole 79M-row warehouse.
- **Excluded:** IDs (grouping only), the LLM provider/model columns, and every column built from the label (`trend_*`, 30-day windows) from the features.

In [2]:
print("Contract receipts live in work/notebooks/w03_data_contract.ipynb (March 2026 partition: grain unique, 31 days, availability counts).")
print("Starter snapshot: 30,000 pages, 32 clients; base rate declining =", BASE)

Contract receipts live in work/notebooks/w03_data_contract.ipynb (March 2026 partition: grain unique, 31 days, availability counts).
Starter snapshot: 30,000 pages, 32 clients; base rate declining = 0.5421


## 3. Methodology

- **Baseline (Week 4):** a transparent CTR-gap rule — impressions × (typical CTR for the page's position band − the page's CTR), position ≤ 20, ≥ 1,000 impressions.
- **Models (Week 5):** logistic regression, depth-3 decision tree, random forest; same folds and metrics as the baseline.
- **Validation (Weeks 5–6):** 5-fold **grouped by client**, out-of-fold scores only; seeds fixed at 42. Metrics: AUC and precision@K, always beside the base rate.
- **Leakage checks (Week 6):** deliberate leak test, random-vs-grouped split comparison, and a strict-timeline feature set that avoids the 90-day columns overlapping the label window.

## 4. Results (vs baseline)

In [3]:
names = ["Random\npicking", "Week-4\nbaseline rule", "Decision tree\n(depth 3)", "Random\nforest", "Logistic\nregression"]
keys  = ["(random picking = base rate)", "W4 baseline (CTR-gap rule)", "decision tree (depth 3)", "random forest", "logistic regression"]
auc = [w5[k]["AUC"] for k in keys]; p50 = [w5[k]["P@50"] for k in keys]
fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.6))
for a, vals, ttl, ref in [(ax[0], auc, "AUC (0.5 = random ordering)", 0.5), (ax[1], p50, "Precision@50 (share of top 50 truly declining)", BASE)]:
    cols = [GREY, RED, BLUE, BLUE, GREEN]
    a.bar(range(5), vals, color=cols); a.axhline(ref, color=INK, ls="--", lw=1)
    for i, v in enumerate(vals): a.text(i, v + 0.012, f"{v:.2f}", ha="center", fontsize=8)
    a.set_xticks(range(5)); a.set_xticklabels(names, fontsize=7); a.set_title(ttl, fontsize=9); a.set_ylim(0.3 if ref < 0.52 else 0.4, 0.9 if ref > 0.52 else 0.8)
save(fig, "fig1_model_vs_baseline")
pd.DataFrame(w5).T.round(3)

,AUC,P@20,P@50,P@200
W4 baseline (CTR-gap rule),0.548,0.530,0.560,0.588
logistic regression,0.656,0.760,0.796,0.783
decision tree (depth 3),0.610,0.610,0.620,0.617
random forest,0.674,0.770,0.772,0.763
(random picking = base rate),0.500,0.542,0.542,0.542


In [4]:
sb = w6["split_before_after"]
lab = ["Logistic", "Random forest"]
rand = [sb["logistic regression | random split (before)"]["AUC"], sb["random forest | random split (before)"]["AUC"]]
grp  = [sb["logistic regression | client-grouped (after)"]["AUC"], sb["random forest | client-grouped (after)"]["AUC"]]
fig, ax = plt.subplots(figsize=(5.6, 3.5)); x = np.arange(2); w = 0.34
ax.bar(x - w/2, rand, w, color=GREY, label="random split (pages of a client on both sides)")
ax.bar(x + w/2, grp, w, color=BLUE, label="client-held-out (honest)")
for i in range(2): ax.text(i - w/2, rand[i] + 0.008, f"{rand[i]:.2f}", ha="center", fontsize=8); ax.text(i + w/2, grp[i] + 0.008, f"{grp[i]:.2f}", ha="center", fontsize=8)
ax.axhline(0.5, color=INK, ls="--", lw=1); ax.set_xticks(x); ax.set_xticklabels(lab); ax.set_ylim(0.45, 0.85); ax.set_ylabel("AUC"); ax.legend(fontsize=7, loc="upper right")
save(fig, "fig2_split_before_after")
print(rand, grp)

[0.715, 0.7734] [0.6565, 0.6726]


In [5]:
d = w7["decline_rate_by_ordering"]; acts = ["refresh_content", "protect_page_one", "rewrite_title_meta"]
a1 = [d["ranked by clicks at stake (size)"][a] for a in acts]; a2 = [d["ranked by p_decline (refresh/protect)"][a] for a in acts]
fig, ax = plt.subplots(figsize=(6.2, 3.5)); x = np.arange(3); w = 0.34
ax.bar(x - w/2, a1, w, color=GREY, label="ranked by page size (clicks at stake)")
ax.bar(x + w/2, a2, w, color=BLUE, label="ranked by model probability")
for i in range(3): ax.text(i - w/2, a1[i] + 0.015, f"{a1[i]:.2f}", ha="center", fontsize=8); ax.text(i + w/2, a2[i] + 0.015, f"{a2[i]:.2f}", ha="center", fontsize=8)
ax.axhline(BASE, color=INK, ls="--", lw=1); ax.text(2.45, BASE + 0.015, "base rate 0.54", ha="right", fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(["refresh\n(top 20)", "protect page one\n(top 10)", "rewrite title/meta\n(top 20)"], fontsize=8)
ax.set_ylim(0, 1.15); ax.set_ylabel("share of queued pages declining"); ax.legend(fontsize=7, loc="upper right", ncol=1)
save(fig, "fig3_ordering_negative_result")
print(a1, a2)

[0.25, 0.1, 0.45] [0.85, 0.4, 0.45]


In [6]:
shutil.copy("work/figures/decay_by_staleness_and_age.png", "docs/img/fig4_decay_by_staleness_and_age.png")
print("figures:", sorted(os.listdir("docs/img")))

figures: ['fig1_model_vs_baseline.png', 'fig2_split_before_after.png', 'fig3_ordering_negative_result.png', 'fig4_decay_by_staleness_and_age.png']


## 5. Limitations

One 90-day snapshot of active pages (≥ 1 impression, age ≥ 90 days), 32 clients; no time-ordered test; modest signal (AUC about 0.66, fold spread 0.03–0.07); the 90-day rule inputs overlap the 30-day label window; clicks at stake is an estimate; observational only. The warehouse was used for a one-month contract check and demonstration, not for the main model.

## 6. Ranked recommendations (from the Week-7 playbook)

1. **`refresh_content`** — pages with ≥ 90 days since update and a high model probability of decline; rank by probability.
2. **`rewrite_title_meta`** — pages in positions 0–20 with a CTR below the norm for their position band; rank by estimated missed clicks.
3. **`protect_page_one`** — page-one pages with a high probability of decline; human-review only (small sample, below base rate in the check).
4. `monitor` and `no_action_low_data` — no action.

Do not automate publishing, deleting, redirecting or bulk rewriting.

In [7]:
q = pd.read_csv("work/outputs/action_queue.csv") if os.path.exists("work/outputs/action_queue.csv") else None
print("queue rows:", None if q is None else len(q), "| sprint slots:", w7["sprint_queue_quota"])
print("action counts (30,000 pages):", w7["action_counts"])

queue rows: 9972 | sprint slots: {'rewrite_title_meta': 20, 'refresh_content': 20, 'protect_page_one': 10}
action counts (30,000 pages): {'monitor': 10114, 'no_action_low_data': 9914, 'rewrite_title_meta': 6574, 'refresh_content': 2695, 'protect_page_one': 703}


## 7. Artifacts the paper embeds

Charts regenerated above and copied to `docs/img/` (the deployed page): `fig1_model_vs_baseline`, `fig2_split_before_after`, `fig3_ordering_negative_result`, `fig4_decay_by_staleness_and_age`. Receipts: `work/outputs/*.json`. Notebooks: `w03`–`w07`.

## Self-check

- [x] Every section filled; charts regenerated from committed metrics
- [x] No client names, URLs or private queries; careful claim language
- [x] Seeds fixed (42); no values typed in by hand — everything is read from `work/outputs/*.json`

## 8. Five-minute demo outline

| min | beat | what I say / show |
|---|---|---|
| 0:00-0:45 | **Question** | A content team can open about fifty pages per sprint. Which fifty? Today a hand-written rule such as "CTR is low for its position" decides. Does a learned ranking beat it, and by how much on clients it has never seen? |
| 0:45-1:45 | **Data** | 30,000 pseudonymised pages, 32 clients, a 90-day window; plus a one-month check of the 79M-row warehouse (grain unique, 4.2% analytics coverage). I left the final month sealed. |
| 1:45-3:00 | **Method** | Target: impressions fell by more than 20% in the last 30 days. Baseline: the CTR-gap rule. Models: logistic regression, tree, forest. Validation grouped by client. Then I attacked my own work: a deliberate leak (AUC 0.998), a random-vs-held-out split, and a strict timeline check. |
| 3:00-3:50 | **One chart** | Figure 2: the same models score AUC 0.72-0.77 on a random split but 0.66-0.67 on held-out clients. The gap is how much a careless split would have flattered me. |
| 3:50-4:30 | **One honest result** | Held-out logistic regression: AUC 0.66, precision@50 of 0.80 against a 0.54 base rate. The CTR rule: AUC 0.55. Modest and directional, from one snapshot with no time-ordered test. Also: ranking the refresh queue by page size did worse than random (25% vs 54%). |
| 4:30-5:00 | **One recommendation** | Use the score to order a human review queue for refresh, ranked by probability, never to publish, delete or rewrite automatically. Re-check precision@50 against the base rate monthly. |

Likely question: *Why not just trust the 0.80?* Because pages from the same client share habits, fold base rates range from 0.38 to 0.65, and it is one 90-day snapshot; differences under about 0.03 are noise.

## 9. Two shareable cuts

**Social post (methodology):**

> I built a model to rank which web pages to review first, then spent as long trying to break it as building it.
> A random train/test split scored AUC 0.72. Splitting by client (so the model never sees the site it is tested on) dropped it to 0.66. That gap is what a careless split would have hidden.
> I also added a leaky column on purpose (AUC jumped to 0.998) to prove my test could catch it, and found that ranking by page size picked pages that were LESS likely to decline than chance.
> Takeaway: the honest number is smaller, and still beats the simple CTR rule (0.55). Method, code and limits are in the write-up: https://hamzaviour.github.io/flyrank-ml-internship/

**Employer 3-sentencer:**

> I built and audited a page-level decline-risk ranking for search content, using a 30,000-page pseudonymised snapshot from the FlyRank ML Internship dataset and a one-month, 9.8M-row check of its 79M-row warehouse. Validated on clients it had never seen, logistic regression reached AUC 0.66 and precision@50 of 0.80 against a 0.54 base rate, beating a hand-written CTR rule (AUC 0.55), while a random split would have overstated skill by 0.06-0.10 AUC. I turned that into a human-reviewed action playbook with reason codes, monitoring triggers and a list of what not to automate, and published it as a research page with all limits stated.